In [ ]:
!pip install SimpleITK

In [ ]:
from __future__ import annotations
import os
import math
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import pandas as pd
import SimpleITK as sitk
import numpy as np
import re

In [ ]:
def load_metadata_csv(csv_path: str, sep: str = ';') -> pd.DataFrame:
    """
    CSV format:
    Gender;Birthday;Age;Biopsy positive;CT;PET;Code
    """
    df = pd.read_csv(csv_path, sep=sep, dtype=str)
    if 'Code' not in df.columns:
        raise ValueError("Code column missing")

    df['Code'] = df['Code'].astype(str).str.strip()
    df = df.dropna(subset=['Code']).drop_duplicates(subset=['Code'])

    return df

In [ ]:
def add_expected_paths(df: pd.DataFrame, ct_dir: str, mask_dir: str) -> pd.DataFrame:
    """
    Adds paths to dataframe
    """
    def ct_path(code: str) -> str:
        return os.path.join(ct_dir, f"{code}_CT.nii.gz")

    def mask_path(code: str) -> str:
        return os.path.join(mask_dir, f"{code}_segmentation_ct.nrrd")

    df = df.copy()
    df['ct_path'] = df['Code'].apply(ct_path)
    df['mask_path'] = df['Code'].apply(mask_path)
    return df

In [ ]:
@dataclass
class CaseData:
    code: str
    ct_img: sitk.Image            # Float32
    mask_img: sitk.Image          # UInt8
    ct_path: str
    mask_path: str

In [ ]:
def read_ct_image(path: str) -> sitk.Image:
    img = sitk.ReadImage(path)
    return sitk.Cast(img, sitk.sitkFloat32)

def read_mask_image(path: str, threshold: float = 0.5) -> sitk.Image:
    img = sitk.ReadImage(path)
    img = sitk.BinaryThreshold(img, lowerThreshold=threshold, upperThreshold=1e9,
                               insideValue=1, outsideValue=0)
    return sitk.Cast(img, sitk.sitkUInt8)

In [ ]:
def load_case(code: str, ct_dir: str, mask_dir: str) -> CaseData:
    ct_path = os.path.join(ct_dir, f"{code}_CT.nii.gz")
    mask_path = os.path.join(mask_dir, f"{code}_segmentation_ct.nrrd")

    ct_img = read_ct_image(ct_path)
    mask_img = read_mask_image(mask_path)

    return CaseData(code=code, ct_img=ct_img, mask_img=mask_img,
                    ct_path=ct_path, mask_path=mask_path)

In [ ]:
def itk_to_numpy(img: sitk.Image) -> np.ndarray:
    """
    Returns NumPy as (z, y, x).
    """
    return sitk.GetArrayFromImage(img)

def numpy_to_itk(arr_zyx: np.ndarray, reference: sitk.Image,
                 pixel_type: int = sitk.sitkFloat32) -> sitk.Image:
    """
    Returns from (z, y, x) array + metadata from reference
    """
    img = sitk.GetImageFromArray(arr_zyx)
    img = sitk.Cast(img, pixel_type)
    img.SetSpacing(reference.GetSpacing())
    img.SetOrigin(reference.GetOrigin())
    img.SetDirection(reference.GetDirection())
    return img

In [ ]:
def compute_new_size(img: sitk.Image, target_spacing: Tuple[float, float, float]) -> Tuple[int, int, int]:
    old_spacing = img.GetSpacing()
    old_size = img.GetSize()
    new_size = []
    for i in range(3):
        new_sz = int(round(old_size[i] * (old_spacing[i] / float(target_spacing[i]))))
        new_sz = max(1, new_sz)
        new_size.append(new_sz)
    return tuple(new_size)

In [ ]:
def resample_image_to_spacing(img: sitk.Image,
                              target_spacing: Tuple[float, float, float],
                              interpolator=sitk.sitkLinear,
                              default_value: float = 0.0) -> sitk.Image:
    """
    Resamples image to target spacing keeping origin, direction and FOV.
    Interpolator:
      - CT: sitk.sitkLinear
      - Mask: sitk.sitkNearestNeighbor
    """
    new_size = compute_new_size(img, target_spacing)

    resampler = sitk.ResampleImageFilter()
    resampler.SetInterpolator(interpolator)
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetDefaultPixelValue(default_value)

    resampler.SetTransform(sitk.Transform())
    out = resampler.Execute(img)
    return out


def resample_to_reference(img: sitk.Image,
                          reference: sitk.Image,
                          interpolator=sitk.sitkLinear,
                          default_value: float = 0.0) -> sitk.Image:

    resampler = sitk.ResampleImageFilter()
    resampler.SetInterpolator(interpolator)
    resampler.SetOutputSpacing(reference.GetSpacing())
    resampler.SetSize(reference.GetSize())
    resampler.SetOutputOrigin(reference.GetOrigin())
    resampler.SetOutputDirection(reference.GetDirection())
    resampler.SetDefaultPixelValue(default_value)
    resampler.SetTransform(sitk.Transform())
    return resampler.Execute(img)

In [ ]:
@dataclass
class ResampledPair:
    ct_resampled: sitk.Image
    mask_resampled: sitk.Image
    target_spacing: Tuple[float, float, float]

def resample_ct_and_mask(ct_img: sitk.Image,
                         mask_img: sitk.Image,
                         target_spacing: Tuple[float, float, float] = (1.0, 1.0, 3.8)) -> ResampledPair:

    ct_r = resample_image_to_spacing(ct_img, target_spacing, interpolator=sitk.sitkLinear, default_value=0.0)
    mask_r = resample_to_reference(mask_img, ct_r, interpolator=sitk.sitkNearestNeighbor, default_value=0)
    return ResampledPair(ct_resampled=ct_r, mask_resampled=mask_r, target_spacing=target_spacing)

In [ ]:
def apply_lung_mask(ct_img: sitk.Image,
                    mask_img: sitk.Image,
                    outside_value: float = -1000.0) -> sitk.Image:
    masker = sitk.MaskImageFilter()
    masker.SetOutsideValue(float(outside_value))
    masked = masker.Execute(ct_img, mask_img)
    return masked

In [ ]:
def get_mask_bounding_box(mask_img: sitk.Image) -> Tuple[Tuple[int,int,int], Tuple[int,int,int]]:
    stats = sitk.LabelShapeStatisticsImageFilter()
    stats.Execute(mask_img)

    if not stats.HasLabel(1):
        print(f"Label 1 not found in mask")
        return (0, 0, 0), mask_img.GetSize()

    bbox = stats.GetBoundingBox(1)  # (x, y, z, size_x, size_y, size_z)
    idx = (bbox[0], bbox[1], bbox[2])
    size = (bbox[3], bbox[4], bbox[5])
    return idx, size

In [ ]:
def expand_bbox_with_margin(idx_xyz: Tuple[int,int,int],
                            size_xyz: Tuple[int,int,int],
                            img: sitk.Image,
                            margin_mm: Tuple[float,float,float] = (5.0, 5.0, 5.0)) -> Tuple[Tuple[int,int,int], Tuple[int,int,int]]:
    """
    Extends bbox with a margin in mm converted to voxels.
    """
    spacing = img.GetSpacing()  # (sx, sy, sz) mm/voxel
    # mm -> voxels
    mx = int(round(margin_mm[0] / spacing[0]))
    my = int(round(margin_mm[1] / spacing[1]))
    mz = int(round(margin_mm[2] / spacing[2]))

    start = [max(0, idx_xyz[0] - mx),
             max(0, idx_xyz[1] - my),
             max(0, idx_xyz[2] - mz)]

    end = [idx_xyz[0] + size_xyz[0] + mx,
           idx_xyz[1] + size_xyz[1] + my,
           idx_xyz[2] + size_xyz[2] + mz]

    img_size = img.GetSize()

    end = [min(img_size[0], end[0]),
           min(img_size[1], end[1]),
           min(img_size[2], end[2])]

    new_size = [end[0] - start[0],
                end[1] - start[1],
                end[2] - start[2]]

    new_size = [max(1, s) for s in new_size]
    return (start[0], start[1], start[2]), (new_size[0], new_size[1], new_size[2])

In [ ]:
def crop_itk_image(img: sitk.Image,
                   start_xyz: Tuple[int,int,int],
                   size_xyz: Tuple[int,int,int]) -> sitk.Image:

    roi = sitk.RegionOfInterest(img,
                                size=size_xyz,
                                index=start_xyz)
    return roi

In [ ]:
@dataclass
class MaskAndCropOutput:
    masked_img: sitk.Image
    cropped_img: Optional[sitk.Image]
    bbox_start_xyz: Tuple[int,int,int]
    bbox_size_xyz: Tuple[int,int,int]

def mask_and_optional_crop(ct_img: sitk.Image,
                           mask_img: sitk.Image,
                           outside_value: float = -1000.0,
                           do_crop: bool = True,
                           margin_mm: Tuple[float,float,float] = (5.0, 5.0, 5.0)) -> MaskAndCropOutput:
    """
    1. Applies mask
    2. If do_crop=True -> crops to mask bounding box with margin in mm.
    """
    masked = apply_lung_mask(ct_img, mask_img, outside_value=outside_value)

    idx, size = get_mask_bounding_box(mask_img)
    cropped = None
    if do_crop:
        start2, size2 = expand_bbox_with_margin(idx, size, masked, margin_mm=margin_mm)
        cropped = crop_itk_image(masked, start2, size2)
        return MaskAndCropOutput(masked_img=masked, cropped_img=cropped,
                                 bbox_start_xyz=start2, bbox_size_xyz=size2)
    else:
        return MaskAndCropOutput(masked_img=masked, cropped_img=None,
                                 bbox_start_xyz=idx, bbox_size_xyz=size)

In [ ]:
def clip_intensity(img: sitk.Image, min_hu: float = -1000.0, max_hu: float = 400.0) -> sitk.Image:
    clipper = sitk.IntensityWindowingImageFilter()
    clipper.SetWindowMinimum(min_hu)
    clipper.SetWindowMaximum(max_hu)
    clipper.SetOutputMinimum(min_hu)
    clipper.SetOutputMaximum(max_hu)
    return clipper.Execute(img)

In [ ]:
from typing import Literal

def normalize_numpy(arr: np.ndarray,
                    method: Literal["minmax", "zscore"] = "minmax",
                    min_hu: float = -1000.0,
                    max_hu: float = 400.0) -> np.ndarray:

    arr = np.clip(arr, min_hu, max_hu)

    if method == "minmax":
        arr_norm = (arr - min_hu) / (max_hu - min_hu)
    elif method == "zscore":
        mask_valid = arr > (min_hu + 1)   # excludes -1000
        mean = arr[mask_valid].mean() if mask_valid.any() else 0.0
        std = arr[mask_valid].std() if mask_valid.any() else 1.0
        arr_norm = (arr - mean) / (std + 1e-8)
    else:
        raise ValueError("Invalid method")

    return arr_norm.astype(np.float32)

In [ ]:
@dataclass
class NormalizedOutput:
    img_normalized: sitk.Image
    method: str
    min_hu: float
    max_hu: float

def normalize_ct_image(img: sitk.Image,
                       method: Literal["minmax", "zscore"] = "minmax",
                       min_hu: float = -1000.0,
                       max_hu: float = 400.0) -> NormalizedOutput:

    # clipping with ITK
    clipped = clip_intensity(img, min_hu=min_hu, max_hu=max_hu)

    # numPy conversion
    arr = sitk.GetArrayFromImage(clipped)  # (z, y, x)

    # normalization
    arr_norm = normalize_numpy(arr, method=method, min_hu=min_hu, max_hu=max_hu)

    # simpleITK conversion
    img_norm = sitk.GetImageFromArray(arr_norm)
    img_norm.CopyInformation(img)

    return NormalizedOutput(img_normalized=img_norm,
                            method=method,
                            min_hu=min_hu,
                            max_hu=max_hu)

In [ ]:
def _exists_nonempty(path: str) -> bool:
    return isinstance(path, str) and os.path.exists(path) and os.path.getsize(path) > 0

def _is_valid_nifti(path: str) -> bool:
    try:
        if not _exists_nonempty(path): return False
        _ = sitk.ReadImage(path)
        return True
    except Exception:
        return False

def _is_valid_npy(path: str) -> bool:
    try:
        if not _exists_nonempty(path): return False
        arr = np.load(path, mmap_mode='r')
        _ = arr.shape
        return True
    except Exception:
        return False

In [ ]:
@dataclass
class PrepConfig:
    target_spacing: Tuple[float, float, float] = (1.0, 1.0, 3.8)
    outside_value: float = -1000.0
    margin_mm: Tuple[float, float, float] = (5.0, 5.0, 5.0)
    do_crop: bool = True
    norm_method: str = "minmax"        # "minmax" or "zscore"
    clip_min_hu: float = -1000.0
    clip_max_hu: float = 400.0



@dataclass
class PrepPaths:
    code: str
    ct_nifti_path: str            # normalized CT (NIfTI)
    ct_npy_path: str              # normalized CT (NumPy .npy)
    mask_nifti_path: str          # resampled mask (NIfTI)
    ct_cropped_nifti_path: Optional[str]
    ct_cropped_npy_path: Optional[str]
    bbox_start_xyz: Tuple[int, int, int]
    bbox_size_xyz: Tuple[int, int, int]

In [ ]:
def prepare_one_case(code: str,
                     ct_dir: str,
                     mask_dir: str,
                     out_dir: str,
                     cfg: PrepConfig = PrepConfig(),
                     overwrite: bool = False) -> PrepPaths:

    os.makedirs(out_dir, exist_ok=True)
    out_ct_dir = os.path.join(out_dir, "ct_nifti")
    out_mask_dir = os.path.join(out_dir, "mask_nifti")
    out_npy_dir = os.path.join(out_dir, "npy")
    out_crop_dir = os.path.join(out_dir, "ct_cropped_nifti")
    out_crop_npy_dir = os.path.join(out_dir, "npy_cropped")
    for d in [out_ct_dir, out_mask_dir, out_npy_dir, out_crop_dir, out_crop_npy_dir]:
        os.makedirs(d, exist_ok=True)

    ct_nifti_path = os.path.join(out_ct_dir, f"{code}_CT_preproc.nii.gz")
    mask_nifti_path = os.path.join(out_mask_dir, f"{code}_mask_resampled.nii.gz")
    ct_npy_path = os.path.join(out_npy_dir, f"{code}_CT_preproc.npy")
    ct_cropped_nifti_path = os.path.join(out_crop_dir, f"{code}_CT_preproc_cropped.nii.gz")
    ct_cropped_npy_path = os.path.join(out_crop_npy_dir, f"{code}_CT_preproc_cropped.npy")

    have_ct_nifti = _is_valid_nifti(ct_nifti_path)
    have_mask_nifti = _is_valid_nifti(mask_nifti_path)
    have_ct_npy = _is_valid_npy(ct_npy_path)
    have_crop_nifti = _is_valid_nifti(ct_cropped_nifti_path)
    have_crop_npy = _is_valid_npy(ct_cropped_npy_path)

    if (have_ct_nifti and have_mask_nifti and have_ct_npy and
        (have_crop_nifti or not cfg.do_crop) and (have_crop_npy or not cfg.do_crop) and not overwrite):
        return PrepPaths(code, ct_nifti_path, ct_npy_path, mask_nifti_path,
                         ct_cropped_nifti_path if cfg.do_crop else None,
                         ct_cropped_npy_path if cfg.do_crop else None,
                         bbox_start_xyz=(0,0,0), bbox_size_xyz=(0,0,0))


    case = load_case(code, ct_dir=ct_dir, mask_dir=mask_dir)

    pair = resample_ct_and_mask(case.ct_img, case.mask_img, target_spacing=cfg.target_spacing)
    mask_std = standardize_lung_mask(pair.mask_resampled)

    mc = mask_and_optional_crop(pair.ct_resampled, mask_std,
                                outside_value=cfg.outside_value,
                                do_crop=cfg.do_crop,
                                margin_mm=cfg.margin_mm)

    norm_full = normalize_ct_image(mc.masked_img,
                                   method=cfg.norm_method,
                                   min_hu=cfg.clip_min_hu,
                                   max_hu=cfg.clip_max_hu)
    ct_norm = norm_full.img_normalized

    ct_cropped_norm = None
    if cfg.do_crop and mc.cropped_img is not None:
        norm_crop = normalize_ct_image(mc.cropped_img,
                                       method=cfg.norm_method,
                                       min_hu=cfg.clip_min_hu,
                                       max_hu=cfg.clip_max_hu)
        ct_cropped_norm = norm_crop.img_normalized

    if overwrite or not have_ct_nifti:
        sitk.WriteImage(ct_norm, ct_nifti_path, useCompression=True)
    if overwrite or not have_mask_nifti:
        sitk.WriteImage(mask_std, mask_nifti_path, useCompression=True)
    if overwrite or not have_ct_npy:
        np.save(ct_npy_path, sitk.GetArrayFromImage(ct_norm))

    if cfg.do_crop and ct_cropped_norm is not None:
        if overwrite or not have_crop_nifti:
            sitk.WriteImage(ct_cropped_norm, ct_cropped_nifti_path, useCompression=True)
        if overwrite or not have_crop_npy:
            np.save(ct_cropped_npy_path, sitk.GetArrayFromImage(ct_cropped_norm))

    return PrepPaths(
        code=code,
        ct_nifti_path=ct_nifti_path,
        ct_npy_path=ct_npy_path,
        mask_nifti_path=mask_nifti_path,
        ct_cropped_nifti_path=ct_cropped_nifti_path if cfg.do_crop else None,
        ct_cropped_npy_path=ct_cropped_npy_path if cfg.do_crop else None,
        bbox_start_xyz=mc.bbox_start_xyz,
        bbox_size_xyz=mc.bbox_size_xyz
    )

In [ ]:
def prepare_dataset_from_csv(csv_path: str,
                             ct_dir: str,
                             mask_dir: str,
                             out_dir: str,
                             code_col: str = "Code",
                             sep: str = ";",
                             cfg: PrepConfig = PrepConfig(),
                             overwrite: bool = False,
                             carry_bbox_from_old: bool = True) -> pd.DataFrame:

    meta = pd.read_csv(csv_path, sep=sep, dtype=str)
    if code_col not in meta.columns:
        raise ValueError(f"CSV must contain '{code_col}' column.")
    meta[code_col] = meta[code_col].astype(str).str.strip()
    meta = meta.dropna(subset=[code_col]).drop_duplicates(subset=[code_col])

    os.makedirs(out_dir, exist_ok=True)
    old_manifest_path = os.path.join(out_dir, "manifest_preproc.csv")
    old_manifest = pd.read_csv(old_manifest_path) if os.path.exists(old_manifest_path) else pd.DataFrame()

    needed_bbox_cols = ["bbox_start_x","bbox_start_y","bbox_start_z","bbox_size_x","bbox_size_y","bbox_size_z"]
    for col in needed_bbox_cols:
        if col not in old_manifest.columns:
            old_manifest[col] = None

    rows = []

    for code in meta[code_col].tolist():
        try:
            paths = prepare_one_case(code, ct_dir, mask_dir, out_dir, cfg, overwrite=overwrite)

            if (not overwrite) and carry_bbox_from_old and paths.bbox_start_xyz == (0,0,0) and not old_manifest.empty:
                old_row = old_manifest[old_manifest["Code"].astype(str).str.strip() == str(code)]
                if not old_row.empty:
                    paths = PrepPaths(
                        code=paths.code,
                        ct_nifti_path=paths.ct_nifti_path,
                        ct_npy_path=paths.ct_npy_path,
                        mask_nifti_path=paths.mask_nifti_path,
                        ct_cropped_nifti_path=paths.ct_cropped_nifti_path,
                        ct_cropped_npy_path=paths.ct_cropped_npy_path,
                        bbox_start_xyz=(
                            int(old_row["bbox_start_x"].iloc[0]) if pd.notna(old_row["bbox_start_x"].iloc[0]) else 0,
                            int(old_row["bbox_start_y"].iloc[0]) if pd.notna(old_row["bbox_start_y"].iloc[0]) else 0,
                            int(old_row["bbox_start_z"].iloc[0]) if pd.notna(old_row["bbox_start_z"].iloc[0]) else 0
                        ),
                        bbox_size_xyz=(
                            int(old_row["bbox_size_x"].iloc[0]) if pd.notna(old_row["bbox_size_x"].iloc[0]) else 0,
                            int(old_row["bbox_size_y"].iloc[0]) if pd.notna(old_row["bbox_size_y"].iloc[0]) else 0,
                            int(old_row["bbox_size_z"].iloc[0]) if pd.notna(old_row["bbox_size_z"].iloc[0]) else 0
                        )
                    )

            rows.append({
                "Code": code,
                "ct_nifti_path": paths.ct_nifti_path,
                "ct_npy_path": paths.ct_npy_path,
                "mask_nifti_path": paths.mask_nifti_path,
                "ct_cropped_nifti_path": paths.ct_cropped_nifti_path,
                "ct_cropped_npy_path": paths.ct_cropped_npy_path,
                "bbox_start_x": paths.bbox_start_xyz[0],
                "bbox_start_y": paths.bbox_start_xyz[1],
                "bbox_start_z": paths.bbox_start_xyz[2],
                "bbox_size_x": paths.bbox_size_xyz[0],
                "bbox_size_y": paths.bbox_size_xyz[1],
                "bbox_size_z": paths.bbox_size_xyz[2],
            })

        except Exception as e:
            rows.append({
                "Code": code,
                "error": repr(e)
            })

    manifest = pd.DataFrame(rows)
    manifest_path = os.path.join(out_dir, "manifest_preproc.csv")
    manifest.to_csv(manifest_path, index=False)
    return manifest

In [ ]:
csv_path = "/content/drive/MyDrive/Colab Notebooks/PatientsData.csv"
ct_dir   = "/content/drive/MyDrive/Colab Notebooks/ct"
mask_dir = "/content/drive/MyDrive/Colab Notebooks/masks_ct"
out_dir  = "/content/drive/MyDrive/Colab Notebooks/processed"

cfg = PrepConfig(
    target_spacing=(1.0, 1.0, 3.8),
    outside_value=-1000.0,
    margin_mm=(5, 5, 5),
    do_crop=True,
    norm_method="minmax",
    clip_min_hu=-1000.0,
    clip_max_hu=400.0
)

manifest = prepare_dataset_from_csv(
    csv_path=csv_path,
    ct_dir=ct_dir,
    mask_dir=mask_dir,
    out_dir=out_dir,
    code_col="Code",
    sep=";",
    cfg=cfg
)

print("Processed patients:", len(manifest))
print(manifest.head())

Processed patients: 25
   Code                                      ct_nifti_path  \
0  2011  /content/drive/MyDrive/Colab Notebooks/process...   
1  3111  /content/drive/MyDrive/Colab Notebooks/process...   
2  4111  /content/drive/MyDrive/Colab Notebooks/process...   
3  5111  /content/drive/MyDrive/Colab Notebooks/process...   
4  7011  /content/drive/MyDrive/Colab Notebooks/process...   

                                         ct_npy_path  \
0  /content/drive/MyDrive/Colab Notebooks/process...   
1  /content/drive/MyDrive/Colab Notebooks/process...   
2  /content/drive/MyDrive/Colab Notebooks/process...   
3  /content/drive/MyDrive/Colab Notebooks/process...   
4  /content/drive/MyDrive/Colab Notebooks/process...   

                                     mask_nifti_path  \
0  /content/drive/MyDrive/Colab Notebooks/process...   
1  /content/drive/MyDrive/Colab Notebooks/process...   
2  /content/drive/MyDrive/Colab Notebooks/process...   
3  /content/drive/MyDrive/Colab Notebooks/p

In [ ]:
def standardize_lung_mask(mask_img: sitk.Image) -> sitk.Image:
    bin_img = sitk.BinaryThreshold(mask_img, lowerThreshold=0.5, upperThreshold=1e9, insideValue=1, outsideValue=0)

    arr = sitk.GetArrayFromImage(bin_img).astype(np.uint8)
    frac_pos = arr.mean()

    if frac_pos > 0.5:
        bin_img = sitk.BinaryNot(bin_img)

    bin_img = sitk.Cast(bin_img, sitk.sitkUInt8)
    bin_img.CopyInformation(mask_img)
    return bin_img